# Models v2: LSTM and Prophet vs baselines

Thin wrapper around `src/lstm.py`, `src/prophet_model.py` and `src/compare.py`. Full training takes ~10 min on 8 CPU cores, so it is off by default; the run writes `predictions_lstm.parquet` and `predictions_prophet.parquet` to `processed/`.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd
from src.data_loader import data_dir, load_features
from src.lstm import run_lstm
from src.prophet_model import run_prophet
from src import compare

PROCESSED = data_dir() / "processed"
RETRAIN = False

In [2]:
if RETRAIN:
    df = load_features()
    run_lstm(df).to_parquet(PROCESSED / "predictions_lstm.parquet", index=False)
    run_prophet(df).to_parquet(PROCESSED / "predictions_prophet.parquet", index=False)

Metrics are computed only on test rows that all four models predict (written to `predictions_v2.csv` / `metrics_v2.csv`).

In [3]:
metrics = compare.main()
metrics.pivot_table(index=["subset", "horizon_h"], columns="model", values="MAE").round(2)

model                         hist_gradient_boosting   lstm  persistence  \
subset             horizon_h                                               
test_all           24                          33.34  32.82        36.15   
                   48                          43.87  42.18        47.06   
test_excl_lockdown 24                          35.68  35.43        39.42   
                   48                          47.38  46.08        51.98   

model                         prophet  
subset             horizon_h           
test_all           24           91.44  
                   48          121.44  
test_excl_lockdown 24           85.25  
                   48          107.26

In [4]:
metrics.pivot_table(index=["subset", "horizon_h"], columns="model", values="RMSE").round(2)

model                         hist_gradient_boosting   lstm  persistence  \
subset             horizon_h                                               
test_all           24                          55.18  55.77        62.96   
                   48                          68.92  68.60        78.80   
test_excl_lockdown 24                          59.67  60.49        68.29   
                   48                          74.70  74.56        86.01   

model                         prophet  
subset             horizon_h           
test_all           24          127.87  
                   48          249.09  
test_excl_lockdown 24          122.63  
                   48          197.80

Rows compared in v2 vs all target-valid test rows in v1.

In [5]:
v1 = pd.read_csv(PROCESSED / "metrics_v1.csv")
key = ["subset", "horizon_h"]
rows = (
    v1[v1["model"] == "persistence"].set_index(key)["n_rows"].rename("v1_test_rows").to_frame()
    .join(metrics[metrics["model"] == "persistence"].set_index(key)["n_rows"].rename("v2_common_rows"))
)
rows.assign(pct=(100 * rows["v2_common_rows"] / rows["v1_test_rows"]).round(1))

,,v1_test_rows,v2_common_rows,pct
subset,horizon_h,,,
test_all,24,841299,618906,73.6
test_excl_lockdown,24,628304,475735,75.7
test_all,48,830031,611632,73.7
test_excl_lockdown,48,621820,471426,75.8


## Verdict (MAE)

**test_all:** LSTM beats both HGB and persistence at 24h (32.82 vs 33.34 / 36.15) and 48h (42.18 vs 43.87 / 47.06), while Prophet beats neither (91.44 at 24h, 121.44 at 48h).

**test_excl_lockdown:** LSTM beats both HGB and persistence at 24h (35.43 vs 35.68 / 39.42) and 48h (46.08 vs 47.38 / 51.98), while Prophet beats neither (85.25 at 24h, 107.26 at 48h).